In [1]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time
import os

print("Libraries imported")

Libraries imported


In [2]:
# Open-Meteo Historical Weather API
# Free, unlimited, no API key needed!

# Boston coordinates
BOSTON_LAT = 42.3601
BOSTON_LON = -71.0589

# Test API call for one day
test_url = (
    f"https://archive-api.open-meteo.com/v1/archive?"
    f"latitude={BOSTON_LAT}&longitude={BOSTON_LON}"
    f"&start_date=2023-04-01&end_date=2023-04-01"
    f"&hourly=temperature_2m,precipitation,windspeed_10m,relativehumidity_2m,weathercode"
    f"&timezone=America/New_York"
)

print("Testing API call...")
print(f"URL: {test_url}\n")

response = requests.get(test_url)
data = response.json()

print("Response structure:")
print(f"Keys: {list(data.keys())}")

print("\nHourly data fields:")
print(f"Fields: {list(data['hourly'].keys())}")

print(f"\nSample - First 5 hours of April 1, 2023:")
sample_df = pd.DataFrame({
    'time': data['hourly']['time'][:5],
    'temp_c': data['hourly']['temperature_2m'][:5],
    'precipitation_mm': data['hourly']['precipitation'][:5],
    'wind_speed_kmh': data['hourly']['windspeed_10m'][:5],
    'humidity_pct': data['hourly']['relativehumidity_2m'][:5],
    'weather_code': data['hourly']['weathercode'][:5]
})

print(sample_df)

Testing API call...
URL: https://archive-api.open-meteo.com/v1/archive?latitude=42.3601&longitude=-71.0589&start_date=2023-04-01&end_date=2023-04-01&hourly=temperature_2m,precipitation,windspeed_10m,relativehumidity_2m,weathercode&timezone=America/New_York

Response structure:
Keys: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly']

Hourly data fields:
Fields: ['time', 'temperature_2m', 'precipitation', 'windspeed_10m', 'relativehumidity_2m', 'weathercode']

Sample - First 5 hours of April 1, 2023:
               time  temp_c  precipitation_mm  wind_speed_kmh  humidity_pct  \
0  2023-04-01T00:00     5.0               0.0             5.3            96   
1  2023-04-01T01:00     5.0               0.0             4.4            98   
2  2023-04-01T02:00     4.4               0.0             2.9            98   
3  2023-04-01T03:00     3.7               0.0             3.4           100   
4  202

In [3]:
# Define the date ranges we need
# Based on our cleaned trip data: Apr 2023 - Dec 2024

START_DATE = "2023-04-01"  # Start of NEW format data
END_DATE = "2024-12-31"    # End of 2024

print(f"Fetching weather data from {START_DATE} to {END_DATE}")

# Calculate total days
start = datetime.strptime(START_DATE, "%Y-%m-%d")
end = datetime.strptime(END_DATE, "%Y-%m-%d")
total_days = (end - start).days + 1

print(f"Total days: {total_days}")
print(f"Total hours: {total_days * 24:,}")

Fetching weather data from 2023-04-01 to 2024-12-31
Total days: 641
Total hours: 15,384


In [ ]:
# Open-Meteo allows fetching multiple months at once
# We'll fetch in 3-month chunks to be safe

def fetch_weather_batch(start_date, end_date):
    """
    Fetch weather data for a date range.

    Args:
        start_date: Start date (YYYY-MM-DD)
        end_date: End date (YYYY-MM-DD)
    Returns:
        DataFrame with hourly weather data
    """
    url = (
        f"https://archive-api.open-meteo.com/v1/archive?"
        f"latitude={BOSTON_LAT}&longitude={BOSTON_LON}"
        f"&start_date={start_date}&end_date={end_date}"
        f"&hourly=temperature_2m,precipitation,windspeed_10m,relativehumidity_2m,weathercode"
        f"&timezone=America/New_York"
    )

    print(f"Fetching {start_date} to {end_date}...", end=" ")

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    data = response.json()

    # Convert to DataFrame
    df = pd.DataFrame({
        'datetime': pd.to_datetime(data['hourly']['time']),
        'temperature_c': data['hourly']['temperature_2m'],
        'precipitation_mm': data['hourly']['precipitation'],
        'wind_speed_kmh': data['hourly']['windspeed_10m'],
        'humidity_pct': data['hourly']['relativehumidity_2m'],
        'weather_code': data['hourly']['weathercode']
    })

    print(f"Got {len(df)} hours")

    return df

# Test with one batch
test_df = fetch_weather_batch("2023-04-01", "2023-06-30")
print(f"\nSample data:")
print(test_df.head())
print(f"\nData types:")
print(test_df.dtypes)

Fetching 2023-04-01 to 2023-06-30... Got 2184 hours

Sample data:
             datetime  temperature_c  precipitation_mm  wind_speed_kmh  \
0 2023-04-01 00:00:00            5.0               0.0             5.3   
1 2023-04-01 01:00:00            5.0               0.0             4.4   
2 2023-04-01 02:00:00            4.4               0.0             2.9   
3 2023-04-01 03:00:00            3.7               0.0             3.4   
4 2023-04-01 04:00:00            3.9               0.0             1.4   

   humidity_pct  weather_code  
0            96             3  
1            98             3  
2            98             3  
3           100             3  
4            99             3  

Data types:
datetime            datetime64[ns]
temperature_c              float64
precipitation_mm           float64
wind_speed_kmh             float64
humidity_pct                 int64
weather_code                 int64
dtype: object


In [ ]:
# Define quarterly date ranges to fetch
date_ranges = [
    ("2023-04-01", "2023-06-30"),  # Q2 2023
    ("2023-07-01", "2023-09-30"),  # Q3 2023
    ("2023-10-01", "2023-12-31"),  # Q4 2023
    ("2024-01-01", "2024-03-31"),  # Q1 2024
    ("2024-04-01", "2024-06-30"),  # Q2 2024
    ("2024-07-01", "2024-09-30"),  # Q3 2024
    ("2024-10-01", "2024-12-31"),  # Q4 2024
]

print("="*60)
print("FETCHING ALL WEATHER DATA")
print("="*60)
print(f"Total batches: {len(date_ranges)}")
print()

all_weather_data = []

for i, (start, end) in enumerate(date_ranges, 1):
    print(f"[{i}/{len(date_ranges)}] ", end="")

    df_batch = fetch_weather_batch(start, end)
    all_weather_data.append(df_batch)

    # Small delay to be polite to API (not required but good practice)
    time.sleep(0.5)

# Combine all batches
df_weather = pd.concat(all_weather_data, ignore_index=True)

print("\n" + "="*60)
print("FETCH COMPLETE")
print("="*60)
print(f"Total weather records: {len(df_weather):,}")
print(f"Date range: {df_weather['datetime'].min()} to {df_weather['datetime'].max()}")
print(f"Missing values:")
print(df_weather.isnull().sum())

FETCHING ALL WEATHER DATA
Total batches: 7

[1/7] Fetching 2023-04-01 to 2023-06-30... Got 2184 hours
[2/7] Fetching 2023-07-01 to 2023-09-30... Got 2208 hours
[3/7] Fetching 2023-10-01 to 2023-12-31... Got 2208 hours
[4/7] Fetching 2024-01-01 to 2024-03-31... Got 2184 hours
[5/7] Fetching 2024-04-01 to 2024-06-30... Got 2184 hours
[6/7] Fetching 2024-07-01 to 2024-09-30... Got 2208 hours
[7/7] Fetching 2024-10-01 to 2024-12-31... Got 2208 hours

FETCH COMPLETE
Total weather records: 15,384
Date range: 2023-04-01 00:00:00 to 2024-12-31 23:00:00
Missing values:
datetime            0
temperature_c       0
precipitation_mm    0
wind_speed_kmh      0
humidity_pct        0
weather_code        0
dtype: int64


In [6]:
# Quick exploration of weather data
print("="*60)
print("WEATHER DATA EXPLORATION")
print("="*60)

# Basic statistics
print("\nStatistical Summary:")
print(df_weather.describe())

# Temperature range
print(f"\nTemperature Range:")
print(f"  Min: {df_weather['temperature_c'].min():.1f}°C")
print(f"  Max: {df_weather['temperature_c'].max():.1f}°C")
print(f"  Mean: {df_weather['temperature_c'].mean():.1f}°C")

# Precipitation
total_precip = df_weather['precipitation_mm'].sum()
rainy_hours = (df_weather['precipitation_mm'] > 0).sum()
print(f"\nPrecipitation:")
print(f"  Total: {total_precip:.1f} mm over period")
print(f"  Rainy hours: {rainy_hours:,} ({rainy_hours/len(df_weather)*100:.1f}%)")

# Wind
print(f"\nWind Speed:")
print(f"  Max: {df_weather['wind_speed_kmh'].max():.1f} km/h")
print(f"  Mean: {df_weather['wind_speed_kmh'].mean():.1f} km/h")

# Most common weather codes
print(f"\nMost Common Weather Conditions:")
print(df_weather['weather_code'].value_counts().head())

WEATHER DATA EXPLORATION

Statistical Summary:
                  datetime  temperature_c  precipitation_mm  wind_speed_kmh  \
count                15384   15384.000000      15384.000000    15384.000000   
mean   2024-02-15 11:30:00      12.513456          0.148713       12.450969   
min    2023-04-01 00:00:00     -14.300000          0.000000        0.000000   
25%    2023-09-08 05:45:00       5.400000          0.000000        8.200000   
50%    2024-02-15 11:30:00      13.000000          0.000000       11.500000   
75%    2024-07-24 17:15:00      19.800000          0.000000       15.900000   
max    2024-12-31 23:00:00      35.700000         16.900000       51.400000   
std                    NaN       9.220774          0.701523        6.046095   

       humidity_pct  weather_code  
count  15384.000000  15384.000000  
mean      72.844774      9.644371  
min       12.000000      0.000000  
25%       60.000000      0.000000  
50%       76.000000      2.000000  
75%       89.000000      

In [7]:
# Add derived features that will be useful for ML
print("\nAdding derived features...")

# Extract date/time components
df_weather['date'] = df_weather['datetime'].dt.date
df_weather['hour'] = df_weather['datetime'].dt.hour
df_weather['day_of_week'] = df_weather['datetime'].dt.dayofweek
df_weather['month'] = df_weather['datetime'].dt.month
df_weather['year'] = df_weather['datetime'].dt.year

# Create categorical features
df_weather['is_precipitation'] = (df_weather['precipitation_mm'] > 0).astype(int)
df_weather['is_cold'] = (df_weather['temperature_c'] < 10).astype(int)  # Below 10°C
df_weather['is_hot'] = (df_weather['temperature_c'] > 25).astype(int)   # Above 25°C

# Temperature feels like (simplified)
# Accounts for wind chill
df_weather['feels_like_c'] = df_weather['temperature_c'] - (df_weather['wind_speed_kmh'] * 0.2)

print(f"Added features. New column count: {len(df_weather.columns)}")
print(f"\nFinal columns: {list(df_weather.columns)}")


Adding derived features...
Added features. New column count: 15

Final columns: ['datetime', 'temperature_c', 'precipitation_mm', 'wind_speed_kmh', 'humidity_pct', 'weather_code', 'date', 'hour', 'day_of_week', 'month', 'year', 'is_precipitation', 'is_cold', 'is_hot', 'feels_like_c']


In [8]:
# Save weather data locally
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
weather_dir = os.path.join(BASE_DIR, "data", "weather")
os.makedirs(weather_dir, exist_ok=True)

# Save as CSV
csv_path = os.path.join(weather_dir, "weather_hourly_2023_2024.csv")
df_weather.to_csv(csv_path, index=False)
print(f"Saved CSV: {csv_path}")

# Save as Parquet (for Spark)
parquet_path = os.path.join(weather_dir, "weather_hourly_2023_2024.parquet")
df_weather.to_parquet(parquet_path, index=False)
print(f"Saved Parquet: {parquet_path}")

print(f"\nWeather data saved to: {weather_dir}")
print(f"Total size: ~{len(df_weather) * 15 / 1024:.1f} KB")

Saved CSV: /Users/anshumansingh/Machine Learning /bluebikes-demand-predictor/data/weather/weather_hourly_2023_2024.csv
Saved Parquet: /Users/anshumansingh/Machine Learning /bluebikes-demand-predictor/data/weather/weather_hourly_2023_2024.parquet

Weather data saved to: /Users/anshumansingh/Machine Learning /bluebikes-demand-predictor/data/weather
Total size: ~225.4 KB


In [9]:
# Show upload command for terminal
gcs_path = "gs://bluebikes-demand-predictor-data/data/weather/"

print("\nTo upload to GCS, run in terminal:")
print("="*60)
print(f"gsutil -m cp {weather_dir}/* {gcs_path}")
print("="*60)


To upload to GCS, run in terminal:
gsutil -m cp /Users/anshumansingh/Machine Learning /bluebikes-demand-predictor/data/weather/* gs://bluebikes-demand-predictor-data/data/weather/
